In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
import torch
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler, DataCollatorWithPadding, TrainingArguments, Trainer
from torch.optim import AdamW
from datasets import load_dataset
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import evaluate

## Load Dataset

In [3]:
from datasets import load_dataset

red_pajama = load_dataset("togethercomputer/RedPajama-Data-V2", 'sample', split="train")

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

In [4]:
red_pajama

Dataset({
    features: ['raw_content', 'doc_id', 'meta', 'quality_signals'],
    num_rows: 1050391
})

In [5]:
red_pajama = red_pajama.train_test_split(test_size=0.3)
red_pajama

DatasetDict({
    train: Dataset({
        features: ['raw_content', 'doc_id', 'meta', 'quality_signals'],
        num_rows: 735273
    })
    test: Dataset({
        features: ['raw_content', 'doc_id', 'meta', 'quality_signals'],
        num_rows: 315118
    })
})

In [6]:
training_samples = red_pajama['train']
validation_samples = red_pajama['test']
training_samples = training_samples.rename_column('raw_content', 'text')
validation_samples = validation_samples.rename_column('raw_content', 'text')
print(training_samples)
print(validation_samples)

Dataset({
    features: ['text', 'doc_id', 'meta', 'quality_signals'],
    num_rows: 735273
})
Dataset({
    features: ['text', 'doc_id', 'meta', 'quality_signals'],
    num_rows: 315118
})


In [7]:
training_samples[0]

{'text': 'TIM CHIPP\nElection results mean focus shifts to holding Abilene ISD accountable after bond passes\nTimothy Chipp\nNearly a week later, the dust has settled on the election. Winners, losers and everyone in between have been decided.\nOne of the biggest winners was the Abilene ISD bond, which picked up a substantial amount of support in the community in one of the most voted-upon bonds in recent history.\nMore:Abilene ISD voters approve $138.7 million Abilene ISD construction bond proposal\nMore:Election reflection from Abilene point of view\nMore than 23,600 votes were cast by the end of Election Day, riding the coattails of a competitive Senate and lieutenant governor race that brought more people than normal to the polls.\nOne of those people was Frank Velasco, a 20-year-old Hardin-Simmons University student who put his 2 cents in on the bond while doing his civic duty Tuesday at Hillcrest Church of Christ.\nVelasco said it would have been "irresponsible of him to not vote"

In [8]:
training_samples.column_names

['text', 'doc_id', 'meta', 'quality_signals']

In [9]:
model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# to help save on gpu space and run this a bit faster we'll load the model in 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [11]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

In [12]:
from peft import LoraConfig
# rank defines the rank of the adapter matrix,
# the higher the rank, the more complex the task it's trying to learn
rank = 128

# the alpha is a scaling factor hyper parameter, basically controls how much our
# adapter will influence the models output, the higher this value
# the more our adapter will overpower the original model weights.
# there is a lot of advice out there for what the alpha value should be
# keeping the alpha at around 2x of what the rank is works for this notebook
alpha = rank*2
peft_config = LoraConfig(
    r=rank,
    lora_alpha=alpha,
    lora_dropout=0.05, # dropout for the lora layers while training, to avoid overfitting
    bias="none",
    task_type="CAUSAL_LM",
    # the target modules defines what types of layers to add lora adapters too, so in the network
    # any model that have a name in this list will have a lora adapter added to it,
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj', 'gate_proj', 'down_proj', 'up_proj']
)

In [13]:
from transformers import TrainingArguments
from trl import SFTTrainer

model_checkpoint_path = "./results/llama-7b"

# an important note is that the loss function isn't defined here,
# it's instead stored as a model parameter for models in hf,
# in the case of llama it is cross entropy loss

# first define some training arguments
training_arguments = TrainingArguments(
    output_dir=model_checkpoint_path,
    optim='adamw_torch', #specify what optimizer we wwant to use, in this case a 8bit version of adamw with pagination.
    per_device_train_batch_size=8, # define the number of samples per training batch
    gradient_accumulation_steps=4, # define how many steps to accumulate gradients,
    log_level='debug',
    eval_strategy = "steps",
    save_strategy='steps', # we'll save a checkpoint every epoch
    logging_steps=8,
    eval_steps=8,
    save_steps=8,
    learning_rate=1e-5, # for llm training we want a fairly high learning rate, 1e-4 is a good starting point but it's worth it to play around with this value
    fp16=True,
    num_train_epochs=4,
    max_steps=120,
    save_total_limit=2,
    warmup_ratio=0.1,
    load_best_model_at_end = True,
    overwrite_output_dir = True,
    lr_scheduler_type='linear',# and set our learning rate decay
)

# now that we have our arguments, we'll use that to create our trainer,
# passing in the model, dataset, peft config, tokenizer, ect
trainer = SFTTrainer(
    model=model,
    train_dataset=training_samples,
    eval_dataset=validation_samples,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

Converting train dataset to ChatML:   0%|          | 0/735273 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/735273 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/735273 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/735273 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/315118 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/315118 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/315118 [00:00<?, ? examples/s]

You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.


[2025-05-08 22:59:47,230] [INFO] [real_accelerator.py:222:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/hans/miniconda3/envs/llm/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/hans/miniconda3/envs/llm/compiler_compat/ld: warning: libstdc++.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/hans/miniconda3/envs/llm/compiler_compat/ld: warning: libm.so.6, needed by /usr/local/cuda/lib64/libcufile.so, not found (try using -rpath or -rpath-link)
/home/hans/miniconda3/envs/llm/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::runtime_error::~runtime_error()@GLIBCXX_3.4'
/home/hans/miniconda3/envs/llm/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `__gxx_personality_v0@CXXABI_1.3'
/home/hans/miniconda3/envs/llm/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.so: undefined reference to `std::ostream::tellp()@GLIBCXX_3.4'
/home/hans/miniconda3/envs/llm/compiler_compat/ld: /usr/local/cuda/lib64/libcufile.

In [14]:
trainer.model.print_trainable_parameters()

trainable params: 319,815,680 || all params: 7,058,231,296 || trainable%: 4.5311


In [15]:
trainer.train()

Currently training with a batch size of: 8
The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: quality_signals, meta, text, doc_id. If quality_signals, meta, text, doc_id are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 735,273
  Num Epochs = 1
  Instantaneous batch size per device = 8
  Total train batch size (w. parallel, distributed & accumulation) = 32
  Gradient Accumulation steps = 4
  Total optimization steps = 120
  Number of trainable parameters = 319,815,680


OutOfMemoryError: CUDA out of memory. Tried to allocate 344.00 MiB. GPU 0 has a total capacity of 23.53 GiB of which 235.44 MiB is free. Including non-PyTorch memory, this process has 23.29 GiB memory in use. Of the allocated memory 22.50 GiB is allocated by PyTorch, and 345.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)